In [1]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [2]:
!pip install rasterio
import os
import numpy as np
import rasterio
from rasterio.plot import show
from glob import glob

In [3]:
from collections import defaultdict
import os
from glob import glob
import numpy as np
import rasterio

# === 1. Configuration ===
folder_path = "/content/drive/MyDrive/SmartSDG-Tunisia/Brut_Data/LST"  # Modify this
output_path = "/content/drive/MyDrive/SmartSDG-Tunisia/Standardized_Data/LST-Standarization"  # Modify this
os.makedirs(output_path, exist_ok=True)

In [5]:
# 2. Charger et classer les rasters par mois (clé = "MM") ===
from collections import defaultdict

rasters_by_month = defaultdict(list)

for f in sorted(glob(os.path.join(folder_path, "*.tif"))):
    filename = os.path.basename(f)
    try:

        # Ex: MODIS_NDVI_Tunisia_2024-04_1km.tif
        parts = filename.replace(".tif", "").split("_")
        print(parts)
        year=parts[2]
        month=int (parts[3])
        rasters_by_month[month].append((year, f))
    except:
        print("❌ Ignored:", filename)

# === 3. Traitement mois par mois ===
for month in sorted(rasters_by_month.keys()):
    print(f"📆 Traitement du mois : {month:02d}")

    # Liste triée des fichiers de ce mois
    files = sorted(rasters_by_month[month], key=lambda x: x[0])
    years = [y for y, _ in files]

    stack = []
    meta = None
    for year, fpath in files:
        with rasterio.open(fpath) as src:
            img = src.read(1).astype(float)
            img[img == src.nodata] = np.nan  # Ignorer nodata
            stack.append(img)
            if meta is None:
                meta = src.meta

    stack = np.stack(stack)  # (n_years, rows, cols)


    # === 4. Z-score pour chaque année du mois courant ===
    for idx, (year, _) in enumerate(files):
        mean = np.nanmean(stack[idx], axis=0)
        std = np.nanstd(stack[idx], axis=0)
        z = (stack[idx] - mean) / std
        meta.update(dtype=rasterio.float32, count=1)

        out_name = f"LST_Zscore_{year}_{month:02d}.tif"
        out_file = os.path.join(output_path, out_name)

        with rasterio.open(out_file, "w", **meta) as dst:
            dst.write(z.astype(np.float32), 1)

        print(f"✅ Sauvegardé : {out_name}")

['LST', 'MonthlyMean', '2000', '02']
['LST', 'MonthlyMean', '2000', '03']
['LST', 'MonthlyMean', '2000', '04']
['LST', 'MonthlyMean', '2000', '05']
['LST', 'MonthlyMean', '2000', '06']
['LST', 'MonthlyMean', '2000', '07']
['LST', 'MonthlyMean', '2000', '08']
['LST', 'MonthlyMean', '2000', '09']
['LST', 'MonthlyMean', '2000', '10']
['LST', 'MonthlyMean', '2000', '11']
['LST', 'MonthlyMean', '2000', '12']
['LST', 'MonthlyMean', '2001', '01']
['LST', 'MonthlyMean', '2001', '02']
['LST', 'MonthlyMean', '2001', '03']
['LST', 'MonthlyMean', '2001', '04']
['LST', 'MonthlyMean', '2001', '05']
['LST', 'MonthlyMean', '2001', '06']
['LST', 'MonthlyMean', '2001', '07']
['LST', 'MonthlyMean', '2001', '08']
['LST', 'MonthlyMean', '2001', '09']
['LST', 'MonthlyMean', '2001', '10']
['LST', 'MonthlyMean', '2001', '11']
['LST', 'MonthlyMean', '2001', '12']
['LST', 'MonthlyMean', '2002', '01']
['LST', 'MonthlyMean', '2002', '02']
['LST', 'MonthlyMean', '2002', '03']
['LST', 'MonthlyMean', '2002', '04']
[